# Sprint 2 — Core ONA, Isolation Score & Interactive Topology

Social Telemetry — Seeing the Silent Teams · Canon EMEA Capstone

**Reads**: `Capstone/data/mock_data_nodes.csv`, `Capstone/data/mock_data_edges.csv` (schema v1.2.0)  
**Writes**: outputs under `Capstone/sprints/sprint2/outputs/`

This notebook is a thin narrative wrapper over the reusable modules in `Capstone/src/` and the one-shot runner `run_sprint2.py`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Resolve Capstone root regardless of notebook launch location.
base = Path.cwd().resolve()
while base.name and base.name != 'Capstone':
    if (base / 'data' / 'mock_data_nodes.csv').exists():
        break
    base = base.parent
print('Capstone root:', base)
sys.path.insert(0, str(base / 'src'))

In [ ]:
from metrics import load_edges_and_nodes, compute_node_metrics, compute_power_user_concentration
from isolation_score import compute_isolation_score
from threshold import derive_threshold, assign_flag_and_tier

nodes, edges = load_edges_and_nodes(base)
print({'nodes': len(nodes), 'edges': len(edges)})
edges.head()

In [ ]:
metrics_df = compute_node_metrics(nodes, edges)
metrics_df.head()

In [ ]:
iso = compute_isolation_score(nodes, metrics_df, edges)
joined = metrics_df.merge(iso, on='EMP_ID').merge(
    nodes[['EMP_ID', 'Team', 'Seniority', 'Years_Exp', 'Profile_Type']], on='EMP_ID'
)
joined.groupby('Profile_Type')['Isolation_Score'].agg(['mean', 'median', 'count']).round(4)

In [ ]:
res = derive_threshold(joined['Isolation_Score'], joined['Profile_Type'])
print(f"AUC                  = {res.auc:.4f}")
print(f"tau_ROC              = {res.tau_roc:.4f}")
print(f"Sensitivity (TPR)    = {res.sensitivity:.4f}")
print(f"Specificity (TNR)    = {res.specificity:.4f}")
print(f"TP/FP/TN/FN          = {res.tp}/{res.fp}/{res.tn}/{res.fn}")
print(f"p50 / p75            = {res.p50:.4f} / {res.p75:.4f}")

flagged = assign_flag_and_tier(joined, res)
flagged['Isolation_Risk_Tier'].value_counts()

In [ ]:
conc = compute_power_user_concentration(metrics_df)
conc

In [ ]:
# Team-to-Team directed block density matrix (Cross & Parker 2004 block density)
# Answers Martin's question: "Is one team genuinely a silo?" with hard numbers.
from team_density import compute_team_density, render_heatmap

density_all = compute_team_density(nodes, edges, interaction_type=None, weighted=False)
print('Unweighted directed density matrix (rows=source team, cols=target team):')
display(density_all.round(4))

# Diagonal vs off-diagonal row-mean ratio (>1 means silo pattern)
diag = pd.Series([density_all.loc[t, t] for t in density_all.index], index=density_all.index, name='Intra')
off = pd.Series(
    [(density_all.loc[t].sum() - density_all.loc[t, t]) / (len(density_all.columns) - 1) for t in density_all.index],
    index=density_all.index, name='Off-diag mean',
)
summary = pd.concat([diag, off, (diag/off).rename('Ratio')], axis=1).round(4)
print('\nSilo signature summary (Ratio > 1 => internally cohesive / externally isolated):')
display(summary)

# Inline heatmap rendering (also saved to outputs/)
heatmap_path = base / 'sprints' / 'sprint2' / 'outputs' / 'sprint2_team_density_heatmap.png'
if heatmap_path.exists():
    from IPython.display import Image as IPImage
    display(IPImage(filename=str(heatmap_path)))
else:
    print('Run `python sprints/sprint2/run_sprint2.py` first to produce the heatmap PNG.')

In [ ]:
# Embed the interactive topology inline (assumes WS7 output already generated).
from IPython.display import IFrame, display

html_path = base / 'sprints' / 'sprint2' / 'outputs' / 'sprint2_interactive_topology.html'
if html_path.exists():
    display(IFrame(src=str(html_path), width='100%', height=820))
else:
    print('Run `python sprints/sprint2/run_sprint2.py` first to generate the interactive HTML.')

## Sprint 2 Generated Artifacts

All outputs below are produced by a single command:

```bash
python sprints/sprint2/run_sprint2.py
```

- `sprints/sprint2/outputs/sprint2_nodes_with_metrics.csv` — full per-node table
- `sprints/sprint2/outputs/sprint2_silent_individuals_shortlist.md` — Top 20 individuals
- `sprints/sprint2/outputs/sprint2_silent_teams_aggregated.md` — team-level roll-up
- `sprints/sprint2/outputs/sprint2_power_user_concentration.md` — Power User report
- `sprints/sprint2/outputs/sprint2_validation_report.md` — Scenario Injection Testing
- `sprints/sprint2/outputs/sprint2_counter_metrics.md` — Counter-metrics declaration
- `sprints/sprint2/outputs/sprint2_interactive_topology.html` — pyvis interactive map
- `sprints/sprint2/outputs/sprint2_summary_report.md` — headline numbers
- `sprints/sprint2/outputs/sprint2_review_pack.md` — sponsor evaluator checklist
- `sprints/sprint2/outputs/sprint2_summary.json` — machine-readable summary